### Importing Libraries


In [ ]:
import pandas as pd
import numpy as np

### Data and Pre-processing

In [1]:
iskcon = pd.read_excel("ISKCON_Data.xlsx", sheet_name=2)

NameError: ignored

In [ ]:
iskcon.head()

,SOURCE,REVIEW BY,REVIEW DATE,REVIEW SUBJECT,text,REVIEW RATING,REVIEW TYPE
0,Trip Advisor,3612,2014-04-30,To commercial,This reativly new temple was a big hindu versi...,2,NEGATIVE
1,Trip Advisor,9573519851,2015-07-24,?Amazing temple in Bangalore?,Me and my friends enjoyed a lot in ISKCON temp...,5,POSITIVE
2,Trip Advisor,???? ?,2016-07-28,A well maintained temple,Otherworldly vibrations throuout the sanctuary...,5,POSITIVE
3,Trip Advisor,????? ?,2016-08-23,Temple,"ISKCON temple is very good, located in west of...",5,POSITIVE
4,Google + HK HILL,????? Anil,2015-11-27,NaN,This is a very good place to be for all the de...,4,POSITIVE


In [ ]:
iskcon.columns

Index(['SOURCE', 'REVIEW BY', 'REVIEW DATE', 'REVIEW SUBJECT', 'text',
       'REVIEW RATING', 'REVIEW TYPE'],
      dtype='object')

In [ ]:
iskcon.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4641 entries, 0 to 4640
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   SOURCE          4638 non-null   object        
 1   REVIEW BY       4640 non-null   object        
 2   REVIEW DATE     4641 non-null   datetime64[ns]
 3   REVIEW SUBJECT  2766 non-null   object        
 4   text            4637 non-null   object        
 5   REVIEW RATING   4641 non-null   int64         
 6   REVIEW TYPE     4641 non-null   object        
dtypes: datetime64[ns](1), int64(1), object(5)
memory usage: 253.9+ KB


In [ ]:
iskcon['text'].head()

0    This reativly new temple was a big hindu versi...
1    Me and my friends enjoyed a lot in ISKCON temp...
2    Otherworldly vibrations throuout the sanctuary...
3    ISKCON temple is very good, located in west of...
4    This is a very good place to be for all the de...
Name: text, dtype: object

In [ ]:
iskcon['rating'] = 0

In [ ]:
index_list_positive = iskcon[iskcon['REVIEW RATING'].isin([0,4,5])].index
iskcon.loc[index_list_positive, 'rating'] = 1
iskcon[iskcon['rating'] > 0]

,SOURCE,REVIEW BY,REVIEW DATE,REVIEW SUBJECT,text,REVIEW RATING,REVIEW TYPE,rating
1,Trip Advisor,9573519851,2015-07-24,?Amazing temple in Bangalore?,Me and my friends enjoyed a lot in ISKCON temp...,5,POSITIVE,1
2,Trip Advisor,???? ?,2016-07-28,A well maintained temple,Otherworldly vibrations throuout the sanctuary...,5,POSITIVE,1
3,Trip Advisor,????? ?,2016-08-23,Temple,"ISKCON temple is very good, located in west of...",5,POSITIVE,1
4,Google + HK HILL,????? Anil,2015-11-27,NaN,This is a very good place to be for all the de...,4,POSITIVE,1
5,Facebook,?????? ????? ???? ????????,2016-11-27,NaN,"Amazing, you come here from the comfort of God...",5,POSITIVE,1
...,...,...,...,...,...,...,...,...
4636,Trip Advisor,Yuvraj Agnihotri,2016-12-11,NaN,Experience can't be shared in words.. Just Wow...,5,POSITIVE,1
4637,Trip Advisor,yv_Shastry,2014-12-10,Clean but commercial,The temple is clean and the altitude of the Ha...,4,MIXED,1
4638,Trip Advisor,Yvonne B,2016-01-20,Powerful energy,We were there for chanting and prayers which I...,4,POSITIVE,1
4639,NaN,yyyasssh,2016-12-03,Worship,My god I love Lord Krishna's temple because he...,5,POSITIVE,1


In [ ]:
#index_list_negative = iskcon[iskcon['REVIEW RATING'].isin([1,2,3])].index
#iskcon.loc[index_list_negative, 'rating'] = 0
#iskcon[iskcon['rating'] == 0]

In [ ]:
# Distribution per class
iskcon['rating'].value_counts()/len(iskcon)

1    0.882353
0    0.117647
Name: rating, dtype: float64

In [ ]:
data = iskcon[['text','rating']]
data_no_nan = data.dropna()
data_no_nan['rating'].value_counts()/len(iskcon)

1    0.881491
0    0.117647
Name: rating, dtype: float64

In [ ]:
from sklearn.model_selection import train_test_split
train, valid = train_test_split(data_no_nan, test_size=0.1, random_state=123)

train.iloc[:, 1].value_counts()/len(train)
valid.iloc[:, 1].value_counts()/len(valid)


In [ ]:
train_text = train.iloc[:, 0]
train_class = train.iloc[:, 1]

valid_text = valid.iloc[:, 0]
valid_class = valid.iloc[:, 1]


### Handling Imbalanced Data


In [ ]:
from imblearn.over_sampling import RandomOverSampler
ros = RandomOverSampler(random_state=123)

X_resampled, y_resampled = ros.fit_resample(train[['text']], train['rating'])

from collections import Counter
print(sorted(Counter(y_resampled).items()))

train_resampled = pd.concat([pd.Series(np.squeeze(X_resampled, axis=1)),
                             pd.Series(y_resampled)], axis=1)


[(0, 3680), (1, 3680)]


/usr/local/lib/python3.6/dist-packages/sklearn/externals/six.py:31: FutureWarning: The module is deprecated in version 0.21 and will be removed in version 0.23 since we've dropped support for Python 2.7. Please rely on the official version of six (https://pypi.org/project/six/).
  "(https://pypi.org/project/six/).", FutureWarning)
/usr/local/lib/python3.6/dist-packages/sklearn/utils/deprecation.py:144: FutureWarning: The sklearn.neighbors.base module is  deprecated in version 0.22 and will be removed in version 0.24. The corresponding classes / functions should instead be imported from sklearn.neighbors. Anything that cannot be imported from sklearn.neighbors is now part of the private API.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.6/dist-packages/sklearn/utils/deprecation.py:87: FutureWarning: Function safe_indexing is deprecated; safe_indexing is deprecated in version 0.22 and will be removed in version 0.24.
  warnings.warn(msg, category=FutureWarning)


In [ ]:
# Training and Validation Dataset -- For resampled dataset
train_text = train_resampled.iloc[:, 0]
train_class = train_resampled.iloc[:, 1]

### Model

# New Section

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vect = TfidfVectorizer(stop_words='english', max_features=3000)
X_train_tfidf_vect = tfidf_vect.fit_transform(train_text)
X_train_tfidf_vect.shape


(7360, 3000)

In [ ]:
from sklearn.linear_model import LogisticRegression
clf = LogisticRegression().fit(X_train_tfidf_vect, train_class)

In [ ]:
X_valid_tfidf = tfidf_vect.transform(valid_text) # Notice we are just using transformer and not fit_transform

In [ ]:
predicted = clf.predict(X_valid_tfidf)

In [ ]:
print(metrics.accuracy_score(valid_class, predicted))

0.8232758620689655


In [ ]:
print(metrics.confusion_matrix(valid_class, predicted))

[[ 33  20]
 [ 62 349]]
